# Part A — Big Data Analytics on DDoS Network Traffic

**Student:** Benjamine. S (258762A)
**Batch:** 18  
**Programme:** MSc Artificial Intelligence  
**University:** University of Moratuwa  
**Module:** Big Data Analytics Mini Project

---

## What This Notebook Does

This notebook analyses a real-world DDoS network traffic dataset using Apache Spark.
The dataset is the BCCC-cPacket-Cloud-DDoS-2024, published in March 2024 by York
University and cPacket Networks. It contains 540,494 network flow records with 319
features per flow, covering 17 DDoS attack types and 9 categories of benign traffic.

The analysis answers five questions about the data:

1. How are the attack types distributed — which are common and which are rare?
2. What are the measurable traffic signatures of each attack type?
3. How do timing patterns differ across attack types?
4. Which TCP flag combinations are characteristic of each attack?
5. Which features most clearly separate attack from benign traffic?

---

## Dataset

**Name:** BCCC-cPacket-Cloud-DDoS-2024  
**Source:** York University / cPacket Networks (March 2024)  
**Access:** Kaggle — dhoogla/bccc-cpacket-cloud-ddos-2024  
**License:** CC-BY-SA-4.0  
**Size:** 540,494 flows × 319 features  
**Format:** Parquet (29.5 MB compressed)

## Section 0 — Environment Setup

In [ ]:
# Install PySpark — required in every new Colab session [not needed in VENV]
# !pip install pyspark --quiet

# Core PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F       # column functions: avg, count, round, when, etc.
from pyspark.sql.window import Window        # for window functions like RANK() OVER

# Visualisation — only called on small aggregated results after .toPandas()
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)

print("All libraries imported successfully.")

# --- Create SparkSession ---
spark = SparkSession.builder \
    .appName("BCCC_DDoS_PartA") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# --- Confirm it started ---
print(f"Spark version  : {spark.version}")
print(f"App name       : {spark.sparkContext.appName}")
print(f"Shuffle parts  : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver memory  : {spark.conf.get('spark.driver.memory')}")

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 17.0.19+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.19+0, mixed mode, sharing)

All libraries imported successfully.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 02:11:53 WARN Utils: Your hostname, Benjamines-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.66 instead (on interface en0)
26/05/12 02:11:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 02:11:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version  : 4.0.0
App name       : BCCC_DDoS_PartA
Shuffle parts  : 8
Driver memory  : 4g
